In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# Load data
data = pd.read_csv("Social_Network_Ads.csv")
print("\nFirst five data")
print(data.head())
print("\nData Description")
print(data.describe())

# Check for missing values
print("\nmissing values")
print(data.isnull().sum())

# Feature Engineering
mapped_gender = {"Male":0 , "Female": 1}
data["Gender"] = data["Gender"].map(mapped_gender)

features = [
    "Gender",
    "Age",
    "EstimatedSalary"
]
# Seperate the data
X = data[features]
y = data["Purchased"]



# Split the data
X_train, X_test,y_train, y_test = train_test_split(
    X, y,test_size=0.2, random_state=42)



param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy'
)
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")



best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

print("\nFinal Model Performance:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred):.4f}")

def plot_confusion_matrix(y_true, y_pred, class_names):
    """
    Plot confusion matrix
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()

# Print interpretation
print("\nConfusion Matrix Interpretation:")
print(f"True Negatives (correctly predicted Class 0): {cm[0,0]}")
print(f"False Positives (predicted Class 1, actual Class 0): {cm[0,1]}")
print(f"False Negatives (predicted Class 0, actual Class 1): {cm[1,0]}")
print(f"True Positives (correctly predicted Class 1): {cm[1,1]}")

importance = best_rf.feature_importances_
feature_names = X.columns
sorted_idx = np.argsort(importance)[::-1]

print("\nTop 3 Features:")
for i in range(3):
    print(f"  {i+1}. {features[sorted_idx[i]]}: {importance[sorted_idx[i]]:.4f}")

plot_confusion_matrix(y_test, y_pred, ['Purchase', 'No Purchase'])

# New user data
new_user = pd.DataFrame({
    'Gender': ['Male'],
    'Age': [45],
    'EstimatedSalary': [85000]
})

# Encode gender
new_user['Gender'] = new_user['Gender'].map({'Male': 0, 'Female': 1})

# Predict
prediction = best_rf.predict(new_user)
probability = best_rf.predict_proba(new_user)

print(f"Will Purchase: {'Yes' if prediction[0] == 1 else 'No'}")
print(f"Confidence: {probability[0][1]:.2%}")


First five data
    User ID  Gender  Age  EstimatedSalary  Purchased
0  15624510    Male   19            19000          0
1  15810944    Male   35            20000          0
2  15668575  Female   26            43000          0
3  15603246  Female   27            57000          0
4  15804002    Male   19            76000          0

Data Description
            User ID         Age  EstimatedSalary   Purchased
count  4.000000e+02  400.000000       400.000000  400.000000
mean   1.569154e+07   37.655000     69742.500000    0.357500
std    7.165832e+04   10.482877     34096.960282    0.479864
min    1.556669e+07   18.000000     15000.000000    0.000000
25%    1.562676e+07   29.750000     43000.000000    0.000000
50%    1.569434e+07   37.000000     70000.000000    0.000000
75%    1.575036e+07   46.000000     88000.000000    1.000000
max    1.581524e+07   60.000000    150000.000000    1.000000

missing values
User ID            0
Gender             0
Age                0
EstimatedSalary    

NameError: name 'cm' is not defined